# FlowPure Defense Deep Dive (5 Attack + 5 Benign)

This notebook mirrors the PRADA deep dive workflow, but uses the FlowPure query defense:
- Run a tiny PRADA attack with FlowPure defense
- Run a tiny benign baseline with the same defense
- Show the exact query images sent to the model
- Compare recorded defense scores to direct FlowPure velocity scores
- Visualize the decision boundary with the velocity threshold
- Inspect score dynamics across time inputs t

## Assumptions and Trade-offs

Assumptions in this notebook:
- A trained target checkpoint already exists at TARGET_CHECKPOINT_DIR.
- A flow matching checkpoint exists at FM_CHECKPOINT.
- Analysis is intentionally tiny and limited to the paired query subset available from both runs.

Trade-offs made intentionally:
- The notebook uses `audit_only=True` to observe decisions without interrupting query streams.
- The detector score is the FlowPure paper-style signal: squared velocity norm at t=0.
- PRADA settings are minimized for reproducibility and speed, not extraction quality.
- This notebook focuses on detection behavior only (no purification).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA
from umap import UMAP

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from flowguard.defenses.query.flowpure import FlowPureQueryDefense
from flowguard.experiments.factory import build_experiment_spec
from flowguard.experiments.spec import AttackKind
from flowguard.orchestration import run_experiment

NATURE = {
    "paper": "#F4F1E8",
    "ink": "#2E2A26",
    "moss": "#607D3B",
    "forest": "#355E3B",
    "clay": "#B5654A",
    "bark": "#6D4C41",
    "sky": "#5C7FA3",
    "sand": "#D8C8A8",
    "stone": "#7D8A91",
}


def apply_nature_style() -> None:
    plt.rcParams.update(
        {
            "figure.facecolor": NATURE["paper"],
            "axes.facecolor": NATURE["paper"],
            "savefig.facecolor": NATURE["paper"],
            "axes.edgecolor": NATURE["ink"],
            "axes.labelcolor": NATURE["ink"],
            "text.color": NATURE["ink"],
            "xtick.color": NATURE["ink"],
            "ytick.color": NATURE["ink"],
            "grid.color": "#C7BCA9",
            "grid.alpha": 0.35,
            "axes.grid": True,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "font.family": "DejaVu Serif",
            "figure.dpi": 150,
        }
    )


apply_nature_style()
print(f"Project root: {project_root}")

In [ ]:
RUNS_DIR = project_root / "runs" / "notebook"
DEEP_DIVE_DIR = RUNS_DIR / "flowpure_deep_dive"
DEEP_DIVE_DIR.mkdir(parents=True, exist_ok=True)

FM_CHECKPOINT = project_root / "runs" / "flow_matching" / "cifar10_notebook" / "checkpoint_latest.pt"
FLOWPURE_PGD_CHECKPOINT_CANDIDATES = [
    project_root / "runs" / "flow_matching" / "cifar10_flowpure_pgd" / "checkpoint_latest.pt",
    project_root / "runs" / "flow_matching" / "cifar10_flowpure_pgd_smoke" / "checkpoint_latest.pt",
]
FLOWPURE_PGD_CHECKPOINT = next(
    (path for path in FLOWPURE_PGD_CHECKPOINT_CANDIDATES if path.exists()),
    None,
)
TARGET_CHECKPOINT_DIR = (
    project_root
    / "runs"
    / "notebook"
    / "training-victim-cifar10-vgg16_bn-nodefense"
    / "target_model"
)

SEED_SAMPLES = 16
PRADA_EVAL_N = 128
PRADA_DUPLICATION_ROUNDS = 4
QUERY_BUDGET = SEED_SAMPLES + PRADA_EVAL_N
VIZ_GRID_COUNT = 8
VELOCITY_THRESHOLD = 5000.0

# MAZE / DisGuide are data-free GAN-based extraction attacks. We run a tiny
# configuration through `run_experiment` (like PRADA) and recycle their
# `visualization_records.pt` for the FlowPure detectors below.
MAZE_EVAL_N = 128
DISGUIDE_EVAL_N = 128
MAZE_QUERY_BUDGET = 128
DISGUIDE_QUERY_BUDGET = 128
EXTRACTION_ATTACK_BATCH_SIZE = 16
ATTACK_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], dtype=torch.float32).view(3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010], dtype=torch.float32).view(3, 1, 1)


def assert_preflight() -> None:
    missing: list[Path] = []
    if not FM_CHECKPOINT.exists():
        missing.append(FM_CHECKPOINT)
    if not TARGET_CHECKPOINT_DIR.exists():
        missing.append(TARGET_CHECKPOINT_DIR)
    if missing:
        missing_lines = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(
            "Missing required artifact(s). Update the notebook paths before running experiments:\n"
            f"{missing_lines}"
        )


assert_preflight()
print(f"Flow checkpoint: {FM_CHECKPOINT}")
print(f"Target checkpoint: {TARGET_CHECKPOINT_DIR}")
print(f"Output dir: {DEEP_DIVE_DIR}")
print(f"PRADA duplication rounds: {PRADA_DUPLICATION_ROUNDS}")

In [ ]:
def build_flowpure_parameters() -> dict[str, Any]:
    return {
        "fm_checkpoint_path": str(FM_CHECKPOINT),
        "dataset_name": "CIFAR10",
        "device": ATTACK_DEVICE,
        "velocity_threshold": VELOCITY_THRESHOLD,
        "inputs_normalized": True,
        "audit_only": True,
    }


def build_common_spec_kwargs() -> dict[str, Any]:
    return {
        "dataset": "CIFAR10",
        "target_architecture": "vgg16_bn",
        "substitute_architecture": "vgg16_bn",
        "query_dataset": "CIFAR10",
        "query_budget": QUERY_BUDGET,
        "query_transfer_set_size": None,
        "target_checkpoint_dir": str(TARGET_CHECKPOINT_DIR),
        "target_device": ATTACK_DEVICE,
        "substitute_device": ATTACK_DEVICE,
        "distributed": False,
        "num_workers": 1,
        "num_clients": 1,
        "attack_batch_size": 1,
        "training_batch_size": 16,
        "epochs": 1,
        "verbose": True,
    }


def run_tiny_prada_with_flowpure() -> tuple[Any, Any]:
    prada_extra = {
        "initial_seed_size": SEED_SAMPLES,
        "duplication_rounds": PRADA_DUPLICATION_ROUNDS,
        "hyperparameter_search_budget": 0,
        "verbose": True,
    }
    spec = build_experiment_spec(
        name="flowpure-deep-dive-prada",
        attack_kind=AttackKind.PRADA,
        attack_mode="prada",
        query_defense="flowpure",
        query_defense_parameters=build_flowpure_parameters(),
        attack_extra=prada_extra,
        **build_common_spec_kwargs(),
    )
    result = run_experiment(spec, output_dir=DEEP_DIVE_DIR / spec.name)
    return spec, result


def run_tiny_benign_with_flowpure() -> tuple[Any, Any]:
    spec = build_experiment_spec(
        name="flowpure-deep-dive-benign",
        attack_kind=AttackKind.TRANSFER_SET,
        attack_mode="naive",
        query_defense="flowpure",
        query_defense_parameters=build_flowpure_parameters(),
        **build_common_spec_kwargs(),
    )
    result = run_experiment(spec, output_dir=DEEP_DIVE_DIR / spec.name)
    return spec, result


def build_extraction_spec_kwargs(query_budget: int) -> dict[str, Any]:
    kwargs = build_common_spec_kwargs()
    kwargs["query_budget"] = query_budget
    kwargs["attack_batch_size"] = EXTRACTION_ATTACK_BATCH_SIZE
    return kwargs


def run_tiny_maze_with_flowpure() -> tuple[Any, Any]:
    # Mirrors the compact configuration used in `scripts/evaluate_detection_suite.py`.
    maze_extra = {
        "latent_dim": 16,
        "iter_gen": 1,
        "iter_clone": 2,
        "iter_exp": 1,
        "ndirs": 1,
        "log_iter": 50,
        "disable_pbar": True,
        "verbose": True,
    }
    spec = build_experiment_spec(
        name="flowpure-deep-dive-maze",
        attack_kind=AttackKind.MAZE,
        attack_mode="maze",
        query_defense="flowpure",
        query_defense_parameters=build_flowpure_parameters(),
        attack_extra=maze_extra,
        **build_extraction_spec_kwargs(MAZE_QUERY_BUDGET),
    )
    result = run_experiment(spec, output_dir=DEEP_DIVE_DIR / spec.name)
    return spec, result


def run_tiny_disguide_with_flowpure() -> tuple[Any, Any]:
    disguide_extra = {
        "ensemble_size": 2,
        "latent_dim": 16,
        "g_iter": 1,
        "d_iter": 1,
        "rep_iter": 1,
        "replay_size": 64,
        "epoch_itrs": 3,
        "disable_pbar": True,
        "verbose": True,
    }
    spec = build_experiment_spec(
        name="flowpure-deep-dive-disguide",
        attack_kind=AttackKind.DISGUIDE,
        attack_mode="disguide",
        query_defense="flowpure",
        query_defense_parameters=build_flowpure_parameters(),
        attack_extra=disguide_extra,
        **build_extraction_spec_kwargs(DISGUIDE_QUERY_BUDGET),
    )
    result = run_experiment(spec, output_dir=DEEP_DIVE_DIR / spec.name)
    return spec, result


def load_extraction_queries_as_x01(
    result: Any,
    *,
    max_samples: int,
    device: str,
) -> torch.Tensor:
    """Load a data-free extraction attack's `visualization_records.pt` and
    return the queries in [0, 1] pixel space. MAZE and DisGuide store the
    exact tensor sent to the victim (normalised CIFAR10 space), so we
    reverse the normalisation here for compatibility with the FlowPure
    detectors below (`inputs_normalized=False`)."""
    records_path = Path(result.attack_summary.output_dir) / "visualization_records.pt"
    if not records_path.exists():
        raise FileNotFoundError(f"Missing extraction records at {records_path}")
    records = _load_transfer_like_records(records_path)
    if not records:
        raise RuntimeError(f"Extraction records at {records_path} are empty.")

    take = min(max_samples, len(records))
    stacked = torch.stack(
        [records[i]["query_x"] for i in range(take)],
        dim=0,
    ).to(device)

    mean_t = CIFAR_MEAN.to(device)
    std_t = CIFAR_STD.to(device)
    return (stacked * std_t + mean_t).clamp(0.0, 1.0)


def _ensure_chw(image: torch.Tensor) -> torch.Tensor:
    tensor = image.detach().cpu().float()
    if tensor.ndim == 2:
        tensor = tensor.unsqueeze(0)
    elif tensor.ndim == 3 and tensor.shape[0] not in {1, 3} and tensor.shape[-1] in {1, 3}:
        tensor = tensor.permute(2, 0, 1)
    return tensor.contiguous()


def _to_model_input(image: torch.Tensor) -> torch.Tensor:
    tensor = _ensure_chw(image)
    if tensor.min().item() >= 0.0 and tensor.max().item() > 5.0:
        tensor = tensor / 255.0
        tensor = (tensor - CIFAR_MEAN) / CIFAR_STD
    return tensor


def _load_transfer_like_records(path: Path) -> list[dict[str, Any]]:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(payload, list):
        raise TypeError(f"Expected list payload in {path}, got {type(payload)}")

    records: list[dict[str, Any]] = []
    for index, entry in enumerate(payload):
        if isinstance(entry, dict):
            query_x = torch.as_tensor(entry.get("query_x", entry.get("x")), dtype=torch.float32)
            original_x = torch.as_tensor(entry.get("original_x", query_x), dtype=torch.float32)
            query_index = int(entry.get("query_index", index))
        else:
            query_x = torch.as_tensor(entry[0], dtype=torch.float32)
            original_x = query_x.clone()
            query_index = index

        records.append(
            {
                "query_x": _to_model_input(query_x),
                "original_x": _to_model_input(original_x),
                "query_index": query_index,
            }
        )

    records.sort(key=lambda item: item["query_index"])
    return records


def _extract_scores_and_flags(result: Any) -> tuple[list[float], list[bool]]:
    scores: list[float] = []
    flags: list[bool] = []
    if result.query_summary.history is None:
        return scores, flags

    for record in result.query_summary.history.records:
        raw_scores = record.metadata.get("flowpure_scores", [])
        raw_flags = record.metadata.get("flowpure_blocked", [])
        for score, flag in zip(raw_scores, raw_flags):
            scores.append(float(score))
            flags.append(bool(flag))
    return scores, flags


def to_display_image_target(query_tensor: torch.Tensor) -> np.ndarray:
    image = _ensure_chw(query_tensor)
    if image.min().item() < -0.25 or image.max().item() > 1.25:
        image = image * CIFAR_STD + CIFAR_MEAN
    elif image.min().item() >= 0.0 and image.max().item() > 5.0:
        image = image / 255.0
    image = image.clamp(0.0, 1.0)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    return np.transpose(image.numpy(), (1, 2, 0))

In [ ]:
FORCE_RERUN_EXPERIMENTS = True


def _cached_run_is_fresh(result_name: str) -> bool:
    if result_name not in globals():
        return False
    cached = globals()[result_name]
    total = getattr(cached.query_summary, "total_queries", 0)
    return total >= QUERY_BUDGET


cached_fresh = _cached_run_is_fresh("prada_result") and _cached_run_is_fresh("benign_result")

if FORCE_RERUN_EXPERIMENTS or not cached_fresh:
    if not cached_fresh and ("prada_result" in globals() or "benign_result" in globals()):
        print(
            f"Cached run has fewer queries than QUERY_BUDGET={QUERY_BUDGET}; re-running "
            "PRADA + benign deep-dives to match the new budget."
        )
    prada_spec, prada_result = run_tiny_prada_with_flowpure()
    benign_spec, benign_result = run_tiny_benign_with_flowpure()
else:
    print("Reusing existing prada_result and benign_result from kernel state.")

print("PRADA run:")
print(f"  total queries: {prada_result.query_summary.total_queries}")
print(f"  output dir: {prada_result.attack_summary.output_dir}")
print("Benign run:")
print(f"  total queries: {benign_result.query_summary.total_queries}")
print(f"  output dir: {benign_result.attack_summary.output_dir}")

prada_meta = prada_result.attack_summary.metadata
prada_records_path = Path(prada_meta.get("visualization_records_path", prada_meta.get("transferset_path", "")))
if not prada_records_path.exists():
    raise FileNotFoundError(f"Could not find PRADA records at {prada_records_path}")

benign_records_path = Path(benign_result.attack_summary.metadata.get("transferset_path", ""))
if not benign_records_path.exists():
    raise FileNotFoundError(f"Could not find benign records at {benign_records_path}")

attack_records_all = _load_transfer_like_records(prada_records_path)
benign_records_all = _load_transfer_like_records(benign_records_path)

attack_scores_all, attack_flags_all = _extract_scores_and_flags(prada_result)
benign_scores_all, benign_flags_all = _extract_scores_and_flags(benign_result)

# PRADA queries the target with `SEED_SAMPLES` clean seeds first to bootstrap the
# substitute model. Those seed queries are unmodified CIFAR images and would
# pollute the attack class. Drop them so `attack_records` contains only the
# actual adversarial-patch queries.
attack_records_patched = attack_records_all[SEED_SAMPLES:]
attack_scores_patched = attack_scores_all[SEED_SAMPLES:]
attack_flags_patched = attack_flags_all[SEED_SAMPLES:]

attack_count = min(
    PRADA_EVAL_N,
    len(attack_records_patched),
    len(attack_scores_patched),
    len(attack_flags_patched),
)
benign_count = min(PRADA_EVAL_N, len(benign_records_all), len(benign_scores_all), len(benign_flags_all))
ANALYSIS_QUERY_COUNT = min(attack_count, benign_count)

if ANALYSIS_QUERY_COUNT <= 0:
    raise RuntimeError("No aligned attack/benign records available for analysis.")

if ANALYSIS_QUERY_COUNT < PRADA_EVAL_N:
    print(
        f"Using {ANALYSIS_QUERY_COUNT} paired queries per source "
        f"(requested {PRADA_EVAL_N} patched PRADA queries; "
        f"available patched={attack_count}, benign={benign_count}). "
        f"PRADA produced {len(attack_records_all)} total records "
        f"({SEED_SAMPLES} seeds + {len(attack_records_patched)} patched). "
        f"Increase QUERY_BUDGET if you want more patched queries."
    )

attack_records = attack_records_patched[:ANALYSIS_QUERY_COUNT]
benign_records = benign_records_all[:ANALYSIS_QUERY_COUNT]
attack_scores = attack_scores_patched[:ANALYSIS_QUERY_COUNT]
attack_flags = attack_flags_patched[:ANALYSIS_QUERY_COUNT]
benign_scores = benign_scores_all[:ANALYSIS_QUERY_COUNT]
benign_flags = benign_flags_all[:ANALYSIS_QUERY_COUNT]
ATTACK_OFFSET = ANALYSIS_QUERY_COUNT

combined_records: list[dict[str, Any]] = []
for index in range(ANALYSIS_QUERY_COUNT):
    combined_records.append(
        {
            "source": "benign",
            "query_id": index,
            "query_x": benign_records[index]["query_x"],
            "original_x": benign_records[index]["original_x"],
            "history_score": benign_scores[index],
            "history_blocked": benign_flags[index],
        }
    )
for index in range(ANALYSIS_QUERY_COUNT):
    combined_records.append(
        {
            "source": "attack",
            "query_id": ATTACK_OFFSET + index,
            "query_x": attack_records[index]["query_x"],
            "original_x": attack_records[index]["original_x"],
            "history_score": attack_scores[index],
            "history_blocked": attack_flags[index],
        }
    )

print(f"\nCombined {2 * ANALYSIS_QUERY_COUNT}-query table (first 8 of each source):")
preview = [
    item for item in combined_records
    if item["source"] == "benign" and item["query_id"] < 8
] + [
    item for item in combined_records
    if item["source"] == "attack" and item["query_id"] < ATTACK_OFFSET + 8
]
for item in preview:
    print(
        f"id={item['query_id']:03d} source={item['source']:<6} "
        f"score={item['history_score']:.2f} blocked={item['history_blocked']}"
    )

In [ ]:
grid_count = min(VIZ_GRID_COUNT, ANALYSIS_QUERY_COUNT)

fig, axes = plt.subplots(
    2,
    grid_count,
    figsize=(2.2 * grid_count, 5.4),
    constrained_layout=True,
    squeeze=False,
)

benign_subset = [item for item in combined_records if item["source"] == "benign"][:grid_count]
attack_subset = [item for item in combined_records if item["source"] == "attack"][:grid_count]

for col, item in enumerate(benign_subset):
    axis = axes[0, col]
    axis.imshow(to_display_image_target(item["query_x"]))
    axis.set_title(
        f"Benign #{col + 1}\nscore={item['history_score']:.1f}",
        fontsize=10,
        color=NATURE["forest"],
    )
    border_color = NATURE["clay"] if item["history_blocked"] else NATURE["forest"]
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2.0)
        spine.set_edgecolor(border_color)
    axis.set_xticks([])
    axis.set_yticks([])

for col, item in enumerate(attack_subset):
    axis = axes[1, col]
    axis.imshow(to_display_image_target(item["query_x"]))
    axis.set_title(
        f"Attack #{col + 1}\nscore={item['history_score']:.1f}",
        fontsize=10,
        color=NATURE["clay"],
    )
    border_color = NATURE["clay"] if item["history_blocked"] else NATURE["forest"]
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2.0)
        spine.set_edgecolor(border_color)
    axis.set_xticks([])
    axis.set_yticks([])

axes[0, 0].set_ylabel("Benign\nqueries", fontsize=12)
axes[1, 0].set_ylabel("PRADA\nqueries", fontsize=12)
fig.suptitle(
    f"Images used to query the target model "
    f"(showing {grid_count} of {ANALYSIS_QUERY_COUNT} benign + {grid_count} of {ANALYSIS_QUERY_COUNT} attack)",
    fontsize=14,
)
plt.show()

In [ ]:
flowpure_defense = FlowPureQueryDefense(
    fm_checkpoint_path=str(FM_CHECKPOINT),
    dataset_name="CIFAR10",
    device=ATTACK_DEVICE,
    velocity_threshold=VELOCITY_THRESHOLD,
    inputs_normalized=True,
    audit_only=True,
)

combined_batch = torch.stack([item["query_x"] for item in combined_records], dim=0).float()
prepared_inputs_device = flowpure_defense._prepare_inputs(combined_batch)
direct_scores = flowpure_defense._estimate_velocity_scores(combined_batch).detach().cpu()

with torch.no_grad():
    t_zero = torch.zeros((prepared_inputs_device.shape[0],), device=flowpure_defense.device, dtype=prepared_inputs_device.dtype)
    velocity_t0_device = flowpure_defense.velocity_model(prepared_inputs_device, t_zero)

prepared_inputs = prepared_inputs_device.detach().cpu()
velocity_t0 = velocity_t0_device.detach().cpu()
direct_flags = (direct_scores > VELOCITY_THRESHOLD).tolist()

for idx, item in enumerate(combined_records):
    item["direct_score"] = float(direct_scores[idx].item())
    item["direct_blocked"] = bool(direct_flags[idx])

score_delta = np.array([item["direct_score"] - item["history_score"] for item in combined_records], dtype=np.float64)
print(f"Mean abs(history-direct) score delta: {np.mean(np.abs(score_delta)):.6f}")
print(f"Max abs(history-direct) score delta:  {np.max(np.abs(score_delta)):.6f}")

print("\nScore comparison table:")
for item in combined_records:
    print(
        f"id={item['query_id']:02d} source={item['source']:<6} "
        f"history={item['history_score']:.2f} direct={item['direct_score']:.2f} "
        f"blocked={item['direct_blocked']}"
    )

In [ ]:
query_ids = np.array([item["query_id"] for item in combined_records], dtype=np.int64)
history_scores_np = np.array([item["history_score"] for item in combined_records], dtype=np.float64)
direct_scores_np = np.array([item["direct_score"] for item in combined_records], dtype=np.float64)
sources = np.array([item["source"] for item in combined_records])
direct_flags_np = np.array([item["direct_blocked"] for item in combined_records], dtype=bool)

point_colors = [NATURE["forest"] if src == "benign" else NATURE["clay"] for src in sources]
point_markers = ["X" if flag else "o" for flag in direct_flags_np]

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.6), constrained_layout=True)

for idx in range(len(combined_records)):
    axes[0].scatter(
        query_ids[idx],
        direct_scores_np[idx],
        color=point_colors[idx],
        marker=point_markers[idx],
        s=90,
        edgecolors=NATURE["ink"],
        linewidths=0.35,
    )
axes[0].plot(query_ids, history_scores_np, color=NATURE["sky"], linewidth=1.2, alpha=0.85, label="History score")
axes[0].plot(query_ids, direct_scores_np, color=NATURE["bark"], linewidth=1.2, alpha=0.85, label="Direct score")
axes[0].axhline(VELOCITY_THRESHOLD, color=NATURE["clay"], linestyle="--", linewidth=1.4, label="Decision threshold")
axes[0].set_title("FlowPure OOD decision from velocity norm", fontsize=13)
axes[0].set_xlabel("Query id")
axes[0].set_ylabel("Velocity score")
axes[0].legend(loc="best")

benign_mask = sources == "benign"
attack_mask = sources == "attack"
axes[1].hist(direct_scores_np[benign_mask], bins=12, alpha=0.55, color=NATURE["forest"], label="Benign")
axes[1].hist(direct_scores_np[attack_mask], bins=12, alpha=0.55, color=NATURE["clay"], label="PRADA")
axes[1].axvline(VELOCITY_THRESHOLD, color=NATURE["ink"], linestyle="--", linewidth=1.4, label="Threshold")
axes[1].set_title("Score distribution and threshold", fontsize=13)
axes[1].set_xlabel("Velocity score")
axes[1].set_ylabel("Count")
axes[1].legend(loc="best")

plt.show()

In [ ]:
prepared_inputs_device = prepared_inputs.to(flowpure_defense.device)
time_grid = torch.linspace(0.0, 1.0, 25, device=flowpure_defense.device)
score_curves = []

with torch.no_grad():
    for t in time_grid:
        t_batch = torch.full((prepared_inputs_device.shape[0],), float(t.item()), device=flowpure_defense.device, dtype=prepared_inputs_device.dtype)
        velocity = flowpure_defense.velocity_model(prepared_inputs_device, t_batch)
        score = velocity.flatten(start_dim=1).pow(2).sum(dim=1)
        score_curves.append(score.detach().cpu().numpy())

score_curves_np = np.stack(score_curves, axis=1)
benign_curves = score_curves_np[:ANALYSIS_QUERY_COUNT]
attack_curves = score_curves_np[ANALYSIS_QUERY_COUNT:]

def _curve_stats(curves: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = curves.mean(axis=0)
    std = curves.std(axis=0)
    return mean, std

benign_mean, benign_std = _curve_stats(benign_curves)
attack_mean, attack_std = _curve_stats(attack_curves)

fig, ax = plt.subplots(figsize=(10.4, 5.0), constrained_layout=True)
ax.plot(time_grid.cpu().numpy(), benign_mean, color=NATURE["forest"], linewidth=2.0, label="Benign mean")
ax.fill_between(
    time_grid.cpu().numpy(),
    benign_mean - benign_std,
    benign_mean + benign_std,
    color=NATURE["forest"],
    alpha=0.2,
    linewidth=0,
)
ax.plot(time_grid.cpu().numpy(), attack_mean, color=NATURE["clay"], linewidth=2.0, label="PRADA mean")
ax.fill_between(
    time_grid.cpu().numpy(),
    attack_mean - attack_std,
    attack_mean + attack_std,
    color=NATURE["clay"],
    alpha=0.2,
    linewidth=0,
)
ax.axhline(VELOCITY_THRESHOLD, color=NATURE["ink"], linestyle="--", linewidth=1.3, label="Threshold")
ax.set_title("Velocity score dynamics across time input t", fontsize=14)
ax.set_xlabel("t")
ax.set_ylabel("Squared velocity norm")
ax.legend(loc="best")
plt.show()

In [ ]:
velocity_flat = velocity_t0.flatten(start_dim=1).numpy()
reference_count = 250
gaussian_reference = torch.randn(reference_count, *prepared_inputs.shape[1:], dtype=prepared_inputs.dtype)

with torch.no_grad():
    t_ref = torch.zeros((reference_count,), device=flowpure_defense.device, dtype=prepared_inputs_device.dtype)
    v_ref = flowpure_defense.velocity_model(gaussian_reference.to(flowpure_defense.device), t_ref).detach().cpu()

reference_flat = v_ref.flatten(start_dim=1).numpy()
stacked = np.vstack([velocity_flat, reference_flat])

pca = PCA(n_components=2, random_state=0)
pca_emb = pca.fit_transform(stacked)

umap = UMAP(
    n_components=2,
    random_state=0,
    n_neighbors=max(2, min(15, stacked.shape[0] - 1)),
    min_dist=0.25,
)
umap_emb = umap.fit_transform(stacked)

query_count = velocity_flat.shape[0]
pca_query = pca_emb[:query_count]
pca_ref = pca_emb[query_count:]
umap_query = umap_emb[:query_count]
umap_ref = umap_emb[query_count:]

fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.0), constrained_layout=True)

for axis, title, query_emb, ref_emb in [
    (axes[0], "PCA of v(t=0, x)", pca_query, pca_ref),
    (axes[1], "UMAP of v(t=0, x)", umap_query, umap_ref),
]:
    axis.scatter(ref_emb[:, 0], ref_emb[:, 1], s=10, color=NATURE["stone"], alpha=0.20, label="Gaussian reference")
    for idx, item in enumerate(combined_records):
        color = NATURE["forest"] if item["source"] == "benign" else NATURE["clay"]
        marker = "X" if item["direct_blocked"] else "o"
        axis.scatter(
            query_emb[idx, 0],
            query_emb[idx, 1],
            s=85,
            color=color,
            marker=marker,
            edgecolors=NATURE["ink"],
            linewidths=0.35,
        )
    axis.set_title(title, fontsize=13)
    axis.set_xlabel("Component 1")
    axis.set_ylabel("Component 2")

axes[0].plot([], [], color=NATURE["forest"], marker="o", linestyle="", label="Benign query")
axes[0].plot([], [], color=NATURE["clay"], marker="o", linestyle="", label="PRADA query")
axes[0].plot([], [], color=NATURE["ink"], marker="X", linestyle="", label="Flagged")
axes[0].legend(loc="best")

fig.suptitle("FlowPure feature-space view at t=0", fontsize=15)
plt.show()

# Data-free Extraction Queries: MAZE + DisGuide

In addition to the gradient-based / patch-based attacks above, the model stealing threat model also includes **data-free** extraction attacks that query the target with samples synthesised by an adversarial generator (MAZE, DFME, DisGUIDE, ...). These queries do not come from any natural image pool and their pixel statistics often drift far from CIFAR-10.

We run small-budget configurations of **MAZE** and **DisGUIDE** (matching the setup in `scripts/evaluate_detection_suite.py`) and reuse their `visualization_records.pt` artefacts as additional attacker query streams. The same [0, 1]-pixel tensors are fed into all three detectors below:

1. The paper's FlowPure^PGD detector (Section 3.3).
2. The attack-agnostic FlowPureBoundary detector.
3. The Dual-CNF ensemble (Gauss + Boundary, max-z).

This gives a fair head-to-head comparison of how each FlowPure variant behaves on gradient-based (PGD), boundary (BAM-ES), random-noise (JBDA), patch-based (PRADA), and generator-based (MAZE, DisGuide) queries.

In [ ]:
def _extraction_run_is_fresh(result_name: str, required_queries: int) -> bool:
    if result_name not in globals():
        return False
    cached = globals()[result_name]
    total = getattr(cached.query_summary, "total_queries", 0)
    return total >= required_queries


if FORCE_RERUN_EXPERIMENTS or not _extraction_run_is_fresh("maze_result", MAZE_EVAL_N):
    maze_spec, maze_result = run_tiny_maze_with_flowpure()
else:
    print("Reusing cached maze_result from kernel state.")

if FORCE_RERUN_EXPERIMENTS or not _extraction_run_is_fresh("disguide_result", DISGUIDE_EVAL_N):
    disguide_spec, disguide_result = run_tiny_disguide_with_flowpure()
else:
    print("Reusing cached disguide_result from kernel state.")

x_maze01 = load_extraction_queries_as_x01(
    maze_result,
    max_samples=MAZE_EVAL_N,
    device=ATTACK_DEVICE,
)
x_disguide01 = load_extraction_queries_as_x01(
    disguide_result,
    max_samples=DISGUIDE_EVAL_N,
    device=ATTACK_DEVICE,
)

print(f"MAZE queries: {tuple(x_maze01.shape)}, range [{x_maze01.min().item():.3f}, {x_maze01.max().item():.3f}]")
print(f"DisGuide queries: {tuple(x_disguide01.shape)}, range [{x_disguide01.min().item():.3f}, {x_disguide01.max().item():.3f}]")

maze_grid_count = min(VIZ_GRID_COUNT, int(x_maze01.shape[0]))
disguide_grid_count = min(VIZ_GRID_COUNT, int(x_disguide01.shape[0]))
shared_grid_count = min(maze_grid_count, disguide_grid_count)

if shared_grid_count == 0:
    print("No MAZE/DisGuide queries available for visualization.")
else:
    fig, axes = plt.subplots(
        2,
        shared_grid_count,
        figsize=(2.2 * shared_grid_count, 5.4),
        constrained_layout=True,
        squeeze=False,
    )

    for col in range(shared_grid_count):
        maze_axis = axes[0, col]
        maze_axis.imshow(x_maze01[col].detach().cpu().permute(1, 2, 0).clamp(0.0, 1.0).numpy())
        maze_axis.set_title(f"MAZE #{col + 1}", fontsize=10, color=NATURE["bark"])
        for spine in maze_axis.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2.0)
            spine.set_edgecolor(NATURE["bark"])
        maze_axis.set_xticks([])
        maze_axis.set_yticks([])

        disguide_axis = axes[1, col]
        disguide_axis.imshow(x_disguide01[col].detach().cpu().permute(1, 2, 0).clamp(0.0, 1.0).numpy())
        disguide_axis.set_title(f"DisGuide #{col + 1}", fontsize=10, color=NATURE["sand"])
        for spine in disguide_axis.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2.0)
            spine.set_edgecolor(NATURE["sand"])
        disguide_axis.set_xticks([])
        disguide_axis.set_yticks([])

    axes[0, 0].set_ylabel("MAZE\nqueries", fontsize=12)
    axes[1, 0].set_ylabel("DisGuide\nqueries", fontsize=12)
    fig.suptitle(
        f"Data-free extraction queries (showing {shared_grid_count} samples per attack)",
        fontsize=14,
    )
    plt.show()

# Paper-faithful PGD Detection (Section 3.3)

The sections above use PRADA queries (patch-based synthesis) so the velocity scores are only weakly related to the detection signal described in the paper. To verify the paper's detection claim directly, we now:

1. Pick `N` benign CIFAR-10 test images and craft PGD L∞ adversarial counterparts against the VGG16 victim with `eps = 8/255`, `alpha = 2/255`, `n = 10` (matching Section 0.A.2).
2. To stress-test how attack-specific the FlowPure^PGD detector really is, we additionally evaluate model-stealing query sources that the detector has *never* seen during training:
   - **BAM-ES**: boundary-proximal queries from the attacker-side evolutionary strategy (Biton Dor & Mirsky, 2024), starting from uniform noise.
   - **JBDA-like**: benign images plus coarse random L∞ perturbation as an off-manifold baseline.
   - **PRADA**: patch-based adversarial queries from the deep dive above.
   - **MAZE** / **DisGuide**: data-free extraction attacks with adversarial generators (reused from the MAZE/DisGuide cell above).
3. Score all four sets with `FlowPureQueryDefense._estimate_velocity_scores`, which evaluates `||v_θ(t=0, x)||^2` (Section 3.3).
4. Calibrate the detection threshold at a target FPR on benign scores, then report AUC and TPR@FPR ∈ {1%, 3%, 5%} per attack source.

Important: this signal is only meaningful for a FlowPure^PGD checkpoint (`x_0 = adv`, `x_1 = clean`). The Gaussian CNF checkpoint at `FM_CHECKPOINT` does NOT implement the paper's detector. If no FlowPure^PGD checkpoint is present, the notebook still runs against the Gaussian checkpoint as a control but will print a warning and we expect AUC ≈ 0.5 on PGD.

Reading the table below: a strong gap between PGD and the unseen attacks (BAM-ES, JBDA-like) is exactly the attack-specific failure mode that motivates the FlowPureBoundary section that follows.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader

from defenses import datasets as legacy_datasets
from flowguard.attacks.boundary_es import boundary_es
from flowguard.attacks.pgd import pgd_linf
from flowguard.serving.model_loader import load_legacy_model

PGD_SECTION_N = 128
PGD_EPS = 8.0 / 255.0
PGD_ALPHA = 2.0 / 255.0
PGD_STEPS = 10
PGD_TARGET_FPR = 0.05

BAM_ES_N = 128
BAM_ES_ITERS = 6
BAM_ES_POP = 32
BAM_ES_TOP_K = 8
BAM_ES_GAMMA = 0.1
JBDA_N = 128
JBDA_EPS = 0.2

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

detection_checkpoint = FLOWPURE_PGD_CHECKPOINT if FLOWPURE_PGD_CHECKPOINT is not None else FM_CHECKPOINT
print(f"Detection checkpoint: {detection_checkpoint}")
if FLOWPURE_PGD_CHECKPOINT is None:
    print(
        "NOTE: no FlowPure^PGD checkpoint found. Falling back to the Gaussian "
        "CNF for smoke-test wiring only. Paper-faithful results require a "
        "checkpoint produced by `python -m flowguard.flow_matching.flowpure_training`."
    )

detection_defense = FlowPureQueryDefense(
    fm_checkpoint_path=str(detection_checkpoint),
    dataset_name="CIFAR10",
    device=ATTACK_DEVICE,
    velocity_threshold=VELOCITY_THRESHOLD,
    inputs_normalized=False,
    audit_only=True,
)

victim = load_legacy_model(TARGET_CHECKPOINT_DIR, device=ATTACK_DEVICE)

from torchvision import transforms as _tv_transforms

cifar_test = legacy_datasets.CIFAR10(train=False, transform=_tv_transforms.ToTensor(), download=False)
loader = DataLoader(cifar_test, batch_size=PGD_SECTION_N, shuffle=False)
x_benign01, y_benign = next(iter(loader))
x_benign01 = x_benign01.to(ATTACK_DEVICE)
y_benign = y_benign.to(ATTACK_DEVICE).long()

print(f"Benign batch shape: {tuple(x_benign01.shape)}, range [{x_benign01.min().item():.3f}, {x_benign01.max().item():.3f}]")

x_adv01 = pgd_linf(
    victim.model,
    x_benign01,
    y_benign,
    eps=PGD_EPS,
    alpha=PGD_ALPHA,
    steps=PGD_STEPS,
    mean=CIFAR10_MEAN,
    std=CIFAR10_STD,
    random_start=True,
)

with torch.no_grad():
    mean_t = torch.tensor(CIFAR10_MEAN, device=ATTACK_DEVICE).view(1, 3, 1, 1)
    std_t = torch.tensor(CIFAR10_STD, device=ATTACK_DEVICE).view(1, 3, 1, 1)
    clean_preds = victim.model((x_benign01 - mean_t) / std_t).argmax(dim=1)
    adv_preds = victim.model((x_adv01 - mean_t) / std_t).argmax(dim=1)

victim_clean_acc = float((clean_preds == y_benign).float().mean().item())
victim_adv_acc = float((adv_preds == y_benign).float().mean().item())
print(
    f"Victim clean accuracy: {victim_clean_acc * 100:.2f}% | "
    f"victim accuracy on PGD-adv: {victim_adv_acc * 100:.2f}% (lower = stronger attack)"
)

bam_seed = torch.empty(BAM_ES_N, 3, 32, 32, device=ATTACK_DEVICE)
x_bam01 = boundary_es(
    victim.model,
    bam_seed,
    mean=CIFAR10_MEAN,
    std=CIFAR10_STD,
    iters=BAM_ES_ITERS,
    pop=BAM_ES_POP,
    top_k=BAM_ES_TOP_K,
    gamma=BAM_ES_GAMMA,
    seed="noise",
)
with torch.no_grad():
    bam_conf = torch.softmax(victim.model((x_bam01 - mean_t) / std_t), dim=1).max(dim=1).values
print(
    f"BAM-ES queries: {x_bam01.shape[0]} (max-softmax mean={bam_conf.mean().item():.3f}, "
    f"min={bam_conf.min().item():.3f})"
)

jbda_base = x_benign01[:JBDA_N]
x_jbda01 = (jbda_base + (torch.rand_like(jbda_base) * 2.0 - 1.0) * JBDA_EPS).clamp(0.0, 1.0)
print(f"JBDA-like queries: {x_jbda01.shape[0]} (eps={JBDA_EPS})")

try:
    prada_records = [r for r in combined_records if r.get("source") == "attack"][:PRADA_EVAL_N]
except NameError:
    prada_records = []

if prada_records:
    prada_x_norm = torch.stack(
        [torch.as_tensor(r["query_x"], dtype=torch.float32) for r in prada_records],
        dim=0,
    ).to(ATTACK_DEVICE)
    x_prada01 = (prada_x_norm * std_t + mean_t).clamp(0.0, 1.0)
    print(
        f"PRADA queries from deep-dive run: {x_prada01.shape[0]} "
        f"(requested {PRADA_EVAL_N}, available {sum(1 for r in combined_records if r.get('source') == 'attack')})"
    )
else:
    x_prada01 = None
    print("PRADA queries unavailable (combined_records not populated); "
          "re-run the PRADA deep-dive cell above to include PRADA in this section.")

In [ ]:
pgd_benign_scores = detection_defense._estimate_velocity_scores(x_benign01).detach().cpu().numpy().astype(float)
pgd_threshold = detection_defense.calibrate_threshold(x_benign01, target_fpr=PGD_TARGET_FPR)
print(f"Calibrated threshold at FPR={PGD_TARGET_FPR:.0%}: {pgd_threshold:.4f}")
print(
    f"Benign score stats: mean={pgd_benign_scores.mean():.2f}, "
    f"median={np.median(pgd_benign_scores):.2f}, max={pgd_benign_scores.max():.2f}"
)

pgd_attack_inputs: dict[str, torch.Tensor] = {
    "PGD": x_adv01,
    "BAM-ES": x_bam01,
    "JBDA-like": x_jbda01,
}
if x_prada01 is not None and x_prada01.numel() > 0:
    pgd_attack_inputs["PRADA"] = x_prada01
try:
    pgd_attack_inputs["MAZE"] = x_maze01
except NameError:
    print("NOTE: x_maze01 not populated; re-run the MAZE/DisGuide cell above.")
try:
    pgd_attack_inputs["DisGuide"] = x_disguide01
except NameError:
    print("NOTE: x_disguide01 not populated; re-run the MAZE/DisGuide cell above.")

pgd_section_metrics: dict[str, dict[str, Any]] = {}
for name, x_attack in pgd_attack_inputs.items():
    adv_scores = detection_defense._estimate_velocity_scores(x_attack).detach().cpu().numpy().astype(float)
    y_true = np.concatenate([np.zeros_like(pgd_benign_scores), np.ones_like(adv_scores)])
    y_score = np.concatenate([pgd_benign_scores, adv_scores])
    auc = float(roc_auc_score(y_true, y_score))
    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_score)

    def _tpr_at(target_fpr: float, fpr_arr=fpr_arr, tpr_arr=tpr_arr) -> float:
        feasible = fpr_arr <= target_fpr
        return float(tpr_arr[feasible].max()) if feasible.any() else 0.0

    detections = int(np.sum(adv_scores > pgd_threshold))
    pgd_section_metrics[name] = {
        "auc": auc,
        "tpr_at_1": _tpr_at(0.01),
        "tpr_at_3": _tpr_at(0.03),
        "tpr_at_5": _tpr_at(0.05),
        "detections": detections,
        "total": int(adv_scores.shape[0]),
        "mean_score": float(adv_scores.mean()),
        "median_score": float(np.median(adv_scores)),
        "fpr": fpr_arr,
        "tpr": tpr_arr,
        "scores": adv_scores,
    }

header = (
    f"{'Source':<12} {'N':>4} {'AUC':>6} {'TPR@1%':>8} {'TPR@3%':>8} {'TPR@5%':>8} "
    f"{'Detected':>10} {'MeanScore':>12}"
)
print(header)
print("-" * len(header))
for name, m in pgd_section_metrics.items():
    print(
        f"{name:<12} {m['total']:>4d} {m['auc']:>6.3f} "
        f"{m['tpr_at_1'] * 100:>7.2f}% {m['tpr_at_3'] * 100:>7.2f}% {m['tpr_at_5'] * 100:>7.2f}% "
        f"{m['detections']:>4d}/{m['total']:<4d} {m['mean_score']:>12.2f}"
    )

In [ ]:
pgd_section_colors = {
    "PGD": NATURE["clay"],
    "BAM-ES": NATURE["forest"],
    "JBDA-like": NATURE["sky"],
    "PRADA": NATURE["moss"],
    "MAZE": NATURE["bark"],
    "DisGuide": NATURE["sand"],
}

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), constrained_layout=True)

axes[0].plot([0.0, 1.0], [0.0, 1.0], color=NATURE["stone"], linestyle="--", linewidth=1.0, label="Chance")
for name, m in pgd_section_metrics.items():
    color = pgd_section_colors.get(name, NATURE["bark"])
    axes[0].plot(m["fpr"], m["tpr"], color=color, linewidth=2.0, label=f"{name} (AUC={m['auc']:.3f})")
axes[0].set_xlim(0.0, 1.0)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title(f"FlowPure^PGD detection ROC across attacks", fontsize=13)
axes[0].legend(loc="lower right", fontsize=9)

all_adv = np.concatenate([m["scores"] for m in pgd_section_metrics.values()])
lo = float(min(pgd_benign_scores.min(), all_adv.min()))
hi = float(max(pgd_benign_scores.max(), all_adv.max()))
bins = np.linspace(lo, hi, 30)

axes[1].hist(pgd_benign_scores, bins=bins, alpha=0.55, color=NATURE["bark"], label="Benign")
for name, m in pgd_section_metrics.items():
    color = pgd_section_colors.get(name, NATURE["clay"])
    axes[1].hist(m["scores"], bins=bins, alpha=0.45, color=color, label=name)
axes[1].axvline(
    pgd_threshold,
    color=NATURE["ink"],
    linestyle="--",
    linewidth=1.3,
    label=f"Threshold @FPR={PGD_TARGET_FPR:.0%}",
)
axes[1].set_xlabel("Squared velocity norm ||v(t=0, x)||^2")
axes[1].set_ylabel("Count")
axes[1].set_title("Score distribution: benign vs attack queries", fontsize=13)
axes[1].legend(loc="best")

fig.suptitle(
    f"FlowPure^PGD detector - checkpoint: {detection_checkpoint.name} "
    f"(noise_type={detection_defense.noise_type})",
    fontsize=14,
)
plt.show()

# Attack-agnostic Detection via Boundary-trained CNF

The PGD section above trained a FlowPure variant on a **specific** attack (PGD L∞, eps=8/255). The detector generalises to that attack by construction, but the model stealing threat model covers many different query types (PRADA patches, BAM-style boundary queries, JBDA-like random perturbations, DFME/Knockoff synthetic queries, ...).

This section evaluates a **FlowPureBoundary** checkpoint — a CNF trained with training pairs `(x_0, x_1)` where `x_0` is produced by a BAM-style evolutionary search on the victim's softmax confidence and `x_1` is the clean seed. Training sees **zero** PGD / CW / FGSM gradients.

The detection signal stays identical: squared velocity norm `||v_θ(t=0, x)||^2` at t=0. What changes is the training distribution, which now reflects "low-confidence under the victim" instead of "PGD-perturbed".

We evaluate against five model-stealing query sources:

1. **PRADA** — patch-based adversarial queries already generated earlier in the notebook (reused from `combined_records`).
2. **BAM-ES** — queries generated on-the-fly via the attacker-side evolutionary strategy of Biton Dor & Mirsky (2024), starting from uniform noise.
3. **JBDA-like** — benign samples plus coarse random L∞ perturbation. Simple off-manifold baseline.
4. **MAZE** — data-free GAN-based extraction queries (Kariyappa et al., 2021), reused from the MAZE/DisGuide cell above.
5. **DisGuide** — ensemble-disagreement GAN extraction queries (Rosenthal et al., 2023), reused from the MAZE/DisGuide cell above.

If no FlowPureBoundary checkpoint is present, the cell falls back to the FlowPure^PGD checkpoint (and further to the Gaussian checkpoint) purely for wiring, printing a note.

In [ ]:
from flowguard.attacks.boundary_es import boundary_es

BOUNDARY_BENIGN_N = 128
BOUNDARY_BAM_N = 128
BOUNDARY_JBDA_N = 128
BOUNDARY_JBDA_EPS = 0.2
BOUNDARY_TARGET_FPR = 0.05
BOUNDARY_ES_ITERS = 6
BOUNDARY_ES_POP = 32
BOUNDARY_ES_TOP_K = 8
BOUNDARY_ES_GAMMA = 0.1

FLOWPURE_BOUNDARY_CHECKPOINT_CANDIDATES = [
    project_root / "runs" / "flow_matching" / "cifar10_flowpure_boundary" / "checkpoint_latest.pt",
    project_root / "runs" / "flow_matching" / "cifar10_flowpure_boundary_smoke" / "checkpoint_latest.pt",
]
FLOWPURE_BOUNDARY_CHECKPOINT = next(
    (path for path in FLOWPURE_BOUNDARY_CHECKPOINT_CANDIDATES if path.exists()),
    None,
)

if FLOWPURE_BOUNDARY_CHECKPOINT is not None:
    boundary_detection_checkpoint = FLOWPURE_BOUNDARY_CHECKPOINT
elif FLOWPURE_PGD_CHECKPOINT is not None:
    boundary_detection_checkpoint = FLOWPURE_PGD_CHECKPOINT
    print(
        "NOTE: no FlowPureBoundary checkpoint found. Falling back to the "
        "FlowPure^PGD checkpoint so the pipeline can still run; results in this "
        "section will reflect the PGD-specific detector, not the attack-agnostic one."
    )
else:
    boundary_detection_checkpoint = FM_CHECKPOINT
    print(
        "NOTE: no FlowPureBoundary or FlowPure^PGD checkpoint found. Falling back "
        "to the Gaussian CNF for wiring only - AUC is expected near 0.5."
    )

print(f"Boundary detection checkpoint: {boundary_detection_checkpoint}")

boundary_defense = FlowPureQueryDefense(
    fm_checkpoint_path=str(boundary_detection_checkpoint),
    dataset_name="CIFAR10",
    device=ATTACK_DEVICE,
    velocity_threshold=VELOCITY_THRESHOLD,
    inputs_normalized=False,
    audit_only=True,
)
print(f"Loaded defense with noise_type={boundary_defense.noise_type!r}")


def _victim_accuracy(x01: torch.Tensor, y: torch.Tensor) -> float:
    mean_t = torch.tensor(CIFAR10_MEAN, device=x01.device).view(1, 3, 1, 1)
    std_t = torch.tensor(CIFAR10_STD, device=x01.device).view(1, 3, 1, 1)
    with torch.no_grad():
        preds = victim.model((x01 - mean_t) / std_t).argmax(dim=1)
    return float((preds == y).float().mean().item())


try:
    benign_count = x_benign01.shape[0]
    print(f"Reusing benign batch from PGD section: {benign_count} samples")
except NameError:
    benign_loader = DataLoader(
        legacy_datasets.CIFAR10(train=False, transform=_tv_transforms.ToTensor(), download=False),
        batch_size=BOUNDARY_BENIGN_N,
        shuffle=False,
    )
    x_benign01, y_benign = next(iter(benign_loader))
    x_benign01 = x_benign01.to(ATTACK_DEVICE)
    y_benign = y_benign.to(ATTACK_DEVICE).long()
    print(f"Loaded fresh benign batch: {x_benign01.shape[0]} samples")

try:
    if x_prada01 is not None:
        print(f"Reusing PRADA queries from PGD section: {x_prada01.shape[0]}")
except NameError:
    prada_records = [r for r in combined_records if r.get("source") == "attack"][:PRADA_EVAL_N] \
        if "combined_records" in globals() else []
    if prada_records:
        prada_x_norm = torch.stack(
            [torch.as_tensor(r["query_x"], dtype=torch.float32) for r in prada_records],
            dim=0,
        ).to(ATTACK_DEVICE)
        prada_mean = torch.tensor(CIFAR10_MEAN, device=ATTACK_DEVICE).view(1, 3, 1, 1)
        prada_std = torch.tensor(CIFAR10_STD, device=ATTACK_DEVICE).view(1, 3, 1, 1)
        x_prada01 = (prada_x_norm * prada_std + prada_mean).clamp(0.0, 1.0)
        print(f"PRADA queries reused from deep-dive run: {x_prada01.shape[0]}")
    else:
        x_prada01 = None
        print("PRADA queries unavailable (combined_records not populated); skipping PRADA in this section.")

try:
    print(f"Reusing BAM-ES queries from PGD section: {x_bam01.shape[0]}")
except NameError:
    bam_seed = torch.empty(BOUNDARY_BAM_N, 3, 32, 32, device=ATTACK_DEVICE)
    x_bam01 = boundary_es(
        victim.model,
        bam_seed,
        mean=CIFAR10_MEAN,
        std=CIFAR10_STD,
        iters=BOUNDARY_ES_ITERS,
        pop=BOUNDARY_ES_POP,
        top_k=BOUNDARY_ES_TOP_K,
        gamma=BOUNDARY_ES_GAMMA,
        seed="noise",
    )
    print(f"BAM-ES attacker queries (fresh): {x_bam01.shape[0]}")

try:
    print(f"Reusing JBDA-like queries from PGD section: {x_jbda01.shape[0]}")
except NameError:
    jbda_base = x_benign01[:BOUNDARY_JBDA_N]
    x_jbda01 = (jbda_base + (torch.rand_like(jbda_base) * 2.0 - 1.0) * BOUNDARY_JBDA_EPS).clamp(0.0, 1.0)
    print(f"JBDA-like queries (fresh): {x_jbda01.shape[0]} (eps={BOUNDARY_JBDA_EPS})")

with torch.no_grad():
    mean_t = torch.tensor(CIFAR10_MEAN, device=ATTACK_DEVICE).view(1, 3, 1, 1)
    std_t = torch.tensor(CIFAR10_STD, device=ATTACK_DEVICE).view(1, 3, 1, 1)
    bam_conf = torch.softmax(victim.model((x_bam01 - mean_t) / std_t), dim=1).max(dim=1).values
print(
    f"BAM-ES max-softmax: mean={bam_conf.mean().item():.3f}, "
    f"min={bam_conf.min().item():.3f}, max={bam_conf.max().item():.3f}"
)

benign_acc = _victim_accuracy(x_benign01, y_benign)
print(f"Victim accuracy on benign batch: {benign_acc * 100:.2f}%")

In [ ]:
boundary_benign_scores = boundary_defense._estimate_velocity_scores(x_benign01).detach().cpu().numpy().astype(float)

boundary_threshold = boundary_defense.calibrate_threshold(x_benign01, target_fpr=BOUNDARY_TARGET_FPR)
print(f"Boundary threshold at FPR={BOUNDARY_TARGET_FPR:.0%}: {boundary_threshold:.4f}")
print(
    f"Benign score stats: mean={boundary_benign_scores.mean():.2f}, "
    f"median={np.median(boundary_benign_scores):.2f}, "
    f"max={boundary_benign_scores.max():.2f}"
)

adversarial_sources: dict[str, np.ndarray] = {}
if x_prada01 is not None and x_prada01.numel() > 0:
    adversarial_sources["PRADA"] = (
        boundary_defense._estimate_velocity_scores(x_prada01).detach().cpu().numpy().astype(float)
    )
adversarial_sources["BAM-ES"] = (
    boundary_defense._estimate_velocity_scores(x_bam01).detach().cpu().numpy().astype(float)
)
adversarial_sources["JBDA-like"] = (
    boundary_defense._estimate_velocity_scores(x_jbda01).detach().cpu().numpy().astype(float)
)
try:
    adversarial_sources["MAZE"] = (
        boundary_defense._estimate_velocity_scores(x_maze01).detach().cpu().numpy().astype(float)
    )
except NameError:
    print("NOTE: x_maze01 unavailable for boundary evaluation; run the MAZE/DisGuide cell.")
try:
    adversarial_sources["DisGuide"] = (
        boundary_defense._estimate_velocity_scores(x_disguide01).detach().cpu().numpy().astype(float)
    )
except NameError:
    print("NOTE: x_disguide01 unavailable for boundary evaluation; run the MAZE/DisGuide cell.")

boundary_metrics: dict[str, dict[str, float]] = {}
for name, adv_scores in adversarial_sources.items():
    y_true = np.concatenate([np.zeros_like(boundary_benign_scores), np.ones_like(adv_scores)])
    y_score = np.concatenate([boundary_benign_scores, adv_scores])
    auc = float(roc_auc_score(y_true, y_score))
    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_score)

    def _tpr_at(target_fpr: float, fpr_arr=fpr_arr, tpr_arr=tpr_arr) -> float:
        feasible = fpr_arr <= target_fpr
        return float(tpr_arr[feasible].max()) if feasible.any() else 0.0

    detections = int(np.sum(adv_scores > boundary_threshold))
    boundary_metrics[name] = {
        "auc": auc,
        "tpr_at_1": _tpr_at(0.01),
        "tpr_at_3": _tpr_at(0.03),
        "tpr_at_5": _tpr_at(0.05),
        "detections": detections,
        "total": int(adv_scores.shape[0]),
        "mean_score": float(adv_scores.mean()),
        "median_score": float(np.median(adv_scores)),
        "fpr": fpr_arr,
        "tpr": tpr_arr,
    }

header = (
    f"{'Source':<12} {'N':>4} {'AUC':>6} {'TPR@1%':>8} {'TPR@3%':>8} {'TPR@5%':>8} "
    f"{'Detected':>10} {'MeanScore':>12}"
)
print(header)
print("-" * len(header))
for name, m in boundary_metrics.items():
    print(
        f"{name:<12} {m['total']:>4d} {m['auc']:>6.3f} "
        f"{m['tpr_at_1'] * 100:>7.2f}% {m['tpr_at_3'] * 100:>7.2f}% {m['tpr_at_5'] * 100:>7.2f}% "
        f"{m['detections']:>4d}/{m['total']:<4d} {m['mean_score']:>12.2f}"
    )

In [ ]:
source_colors = {
    "PRADA": NATURE["clay"],
    "BAM-ES": NATURE["forest"],
    "JBDA-like": NATURE["sky"],
    "MAZE": NATURE["bark"],
    "DisGuide": NATURE["sand"],
}

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), constrained_layout=True)

axes[0].plot([0.0, 1.0], [0.0, 1.0], color=NATURE["stone"], linestyle="--", linewidth=1.0, label="Chance")
for name, m in boundary_metrics.items():
    color = source_colors.get(name, NATURE["bark"])
    axes[0].plot(m["fpr"], m["tpr"], color=color, linewidth=2.0, label=f"{name} (AUC={m['auc']:.3f})")
axes[0].set_xlim(0.0, 1.0)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("FlowPureBoundary detection ROC", fontsize=13)
axes[0].legend(loc="lower right", fontsize=9)

all_adv = np.concatenate(list(adversarial_sources.values()))
lo = float(min(boundary_benign_scores.min(), all_adv.min()))
hi = float(max(boundary_benign_scores.max(), all_adv.max()))
bins = np.linspace(lo, hi, 30)

axes[1].hist(boundary_benign_scores, bins=bins, alpha=0.55, color=NATURE["bark"], label="Benign")
for name, scores in adversarial_sources.items():
    color = source_colors.get(name, NATURE["clay"])
    axes[1].hist(scores, bins=bins, alpha=0.45, color=color, label=name)
axes[1].axvline(
    boundary_threshold,
    color=NATURE["ink"],
    linestyle="--",
    linewidth=1.3,
    label=f"Threshold @FPR={BOUNDARY_TARGET_FPR:.0%}",
)
axes[1].set_xlabel("Squared velocity norm ||v(t=0, x)||^2")
axes[1].set_ylabel("Count")
axes[1].set_title("Score distribution: benign vs model-stealing queries", fontsize=13)
axes[1].legend(loc="best")

fig.suptitle(
    f"Attack-agnostic detection - checkpoint: {boundary_detection_checkpoint.name} "
    f"(noise_type={boundary_defense.noise_type})",
    fontsize=14,
)
plt.show()